# Phase 3 · Nepali (Model L) — Fine-tuning & Analysis

Fine-tunes the pre-trained causal Transformer on the synthetic reasoning dataset using **answer-only loss** (cross-entropy on entity name tokens + EOS only, prompt tokens masked with -100), trains for **4 epochs** with cosine LR scheduling, then evaluates Exact Match (EM), Prefix Match (PM), and Char-Set Accuracy against the pre-trained baseline.

Also runs **attention analysis** comparing entropy and mean-distance metrics across all 6 layers.

In [1]:
import sys, json, math, re, time, unicodedata
from pathlib import Path
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import yaml
from tokenizers import Tokenizer as HFTokenizer

# Import the shared Transformer architecture
REPO_ROOT = Path('../..').resolve()
sys.path.insert(0, str(REPO_ROOT))
from transformer.arch import Transformer

LANG       = 'nepali'
LANG_DIR   = Path('..').resolve()
TOK_DIR    = LANG_DIR / 'tokenizer'
DATA_DIR   = LANG_DIR / 'datasets' / 'reasoning'
CONFIG_PATH= LANG_DIR / 'config.yaml'
CKPT_PRE   = LANG_DIR / 'best_checkpoint.pth'
OUT_DIR    = LANG_DIR / 'finetuned'
RESULTS_DIR= LANG_DIR / 'results'

OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f'Device: {device}')
print(f'Data  : {DATA_DIR}')

Device: mps
Data  : /Users/nikhilsai/Documents/Courses/LMA/SelfProject/individual-project-nikk8076/nepali/datasets/reasoning


## 1. Config, Tokenizer & Model

In [2]:
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
print('Config:', cfg)


class TokenizerWrapper:
    """Thin wrapper for HuggingFace BPE tokenizer."""
    def __init__(self, tok):
        self._tok   = tok
        self._vocab = tok.get_vocab()
        self._eos   = self._vocab.get('[EOS]', 3)
        self._pad   = self._vocab.get('[PAD]', 0)

    def encode(self, text):      return self._tok.encode(text).ids
    def decode(self, ids):       return self._tok.decode(ids)
    def get_vocab_size(self):    return self._tok.get_vocab_size()
    def eos_id(self):            return self._eos
    def pad_id(self):            return self._pad
    def id_to_piece(self, i):    return self._tok.id_to_token(i) or ''


tok_path = TOK_DIR / f'nepali_tokenizer.json'
tokenizer = TokenizerWrapper(HFTokenizer.from_file(str(tok_path)))
print(f'Loaded tokenizer from {tok_path.name} (vocab={tokenizer.get_vocab_size():,}, EOS={tokenizer.eos_id()})')


def build_model(cfg, dropout=0.0):
    return Transformer(
        vocab_size=cfg['vocab_size'], d_model=cfg['d_model'],
        n_heads=cfg['n_heads'], d_ff=cfg['d_ff'], n_layers=cfg['n_layers'],
        max_seq_len=cfg['max_seq_len'], dropout=dropout,
        tie_weights=cfg.get('tie_weights', True),
    )


def load_checkpoint(ckpt_path, cfg, device, dropout=0.0):
    model = build_model(cfg, dropout=dropout).to(device)
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt.get('model_state_dict', ckpt), strict=True)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Loaded {Path(ckpt_path).name}  ({n_params/1e6:.2f}M params)')
    return model


# Load pre-trained model with dropout for training
model = load_checkpoint(CKPT_PRE, cfg, device, dropout=cfg.get('dropout', 0.1))

Config: {'vocab_size': 16000, 'd_model': 512, 'n_heads': 8, 'd_ff': 2048, 'n_layers': 6, 'max_seq_len': 512, 'dropout': 0.1, 'tie_weights': True}
Loaded tokenizer from nepali_tokenizer.json (vocab=16,000, EOS=3)


  Loaded best_checkpoint.pth  (27.09M params)


## 2. Dataset & Answer-Only Loss Masking

In [3]:
MAX_LENGTH = 256


class ReasoningDataset(Dataset):
    """
    Answer-only loss: only answer entity token(s) and EOS contribute to loss.
    Prompt tokens are masked with -100.
    """
    def __init__(self, path, tokenizer, max_length=MAX_LENGTH):
        self.examples = []
        skipped = 0
        eos_id = tokenizer.eos_id()
        for line in open(path, encoding='utf-8'):
            ex = json.loads(line)
            p = ex['prompt'].rstrip()
            ans = ex['answer'].strip()
            full_str = p + ' ' + ans
            full_ids = tokenizer.encode(full_str)
            p_ids = tokenizer.encode(p)
            p_len = len(p_ids)

            if len(full_ids) + 1 > max_length:
                skipped += 1; continue

            input_ids = full_ids + [eos_id]
            labels = [-100] * p_len + full_ids[p_len:] + [eos_id]
            self.examples.append((input_ids, labels))
        print(f'  {Path(path).name}: {len(self.examples):,} examples loaded, {skipped} skipped')

    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]


def collate_fn(batch, pad_id=0):
    inputs, labels = zip(*batch)
    max_len = max(len(x) for x in inputs)
    return (
        torch.tensor([x + [pad_id] * (max_len - len(x)) for x in inputs], dtype=torch.long),
        torch.tensor([l + [-100]   * (max_len - len(l)) for l in labels],  dtype=torch.long),
    )


BATCH_SIZE = 32
train_ds = ReasoningDataset(DATA_DIR / 'train.jsonl', tokenizer)
val_ds   = ReasoningDataset(DATA_DIR / 'val.jsonl',   tokenizer)
test_ds  = ReasoningDataset(DATA_DIR / 'test.jsonl',  tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f'Batches -- train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')

  train.jsonl: 16,000 examples loaded, 0 skipped
  val.jsonl: 2,000 examples loaded, 0 skipped
  test.jsonl: 2,000 examples loaded, 0 skipped
Batches -- train: 500, val: 63, test: 63


## 3. Training Loop

In [4]:
# Hyperparameters
LR           = 4e-4
MAX_EPOCHS   = 4
WARMUP_STEPS = int(len(train_loader) * MAX_EPOCHS * 0.05)
MIN_LR_RATIO = 0.05


def cosine_lr(optimizer, warmup_steps, total_steps, min_lr_ratio=MIN_LR_RATIO):
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / max(1, warmup_steps)
        prog = float(step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(min_lr_ratio, 0.5 * (1.0 + math.cos(math.pi * prog)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def compute_loss(model, input_ids, labels):
    """Answer-only cross-entropy loss (labels=-100 are excluded)."""
    logits, _, _ = model(input_ids)
    return F.cross_entropy(
        logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
        labels[:, 1:].contiguous().view(-1),
        ignore_index=-100,
    )


@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    for inp, lbl in loader:
        inp, lbl = inp.to(device), lbl.to(device)
        total += compute_loss(model, inp, lbl).item(); n += 1
    return total / max(n, 1)


# Load pre-saved best checkpoint and logs
ckpt_ft = torch.load(OUT_DIR / 'best_finetuned_checkpoint.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt_ft['model_state_dict'])
with open(OUT_DIR / 'finetune_log.json') as f:
    eval_log = json.load(f)
logs = eval_log['training_history']
print(f'Loaded best checkpoint from epoch {ckpt_ft.get("epoch", "best")}, best val loss = {ckpt_ft.get("val_loss", 0):.4f}')

Loaded best checkpoint from epoch 4, best val loss = 0.1406


## 4. Fine-tuning Loss & Accuracy Curves

In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
eps = [l['epoch'] for l in logs]
ax1.plot(eps, [l['train_loss'] for l in logs], 'o-', label='Train Loss')
ax1.plot(eps, [l['val_loss']   for l in logs], 's-', label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Loss Curve'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(eps, [l['val_em'] * 100 for l in logs], '^-', color='green', label='Val Exact Match %')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Exact Match %')
ax2.set_title('Validation Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'finetuning_curve.png', dpi=150)
plt.show()
print('Saved:', RESULTS_DIR / 'finetuning_curve.png')

Saved: /Users/nikhilsai/Documents/Courses/LMA/SelfProject/individual-project-nikk8076/nepali/results/finetuning_curve.png


/var/folders/_q/gn6yzv0d07l_f_cgh3l13n1h0000gn/T/ipykernel_82733/2779547201.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Evaluation — Exact Match, Prefix Match, Char-Set Accuracy

In [6]:
def normalize(text):
    text = text.replace('\u2581', ' ').replace('▁', ' ').strip()
    text = unicodedata.normalize('NFC', text)
    return re.sub(r'[\.\u0964\?\,\s]+$', '', text).strip()


@torch.no_grad()
def greedy_generate(mdl, prompt_text, max_new=10):
    p = prompt_text.rstrip()
    ids = tokenizer.encode(p)
    if len(ids) > cfg['max_seq_len'] - 15: ids = ids[-(cfg['max_seq_len'] - 15):]
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    generated = []
    for _ in range(max_new):
        if input_ids.shape[1] >= cfg['max_seq_len']: break
        logits, _, _ = mdl(input_ids)
        next_tok = logits[0, -1, :].argmax().item()
        if next_tok == tokenizer.eos_id(): break
        piece = tokenizer.id_to_piece(next_tok)
        if piece in ('.', '\u0964', '?', '\n'): break
        generated.append(next_tok)
        input_ids = torch.cat(
            [input_ids, torch.tensor([[next_tok]], device=device)], dim=1)
    return normalize(tokenizer.decode(generated))


def evaluate(mdl, test_examples, label=''):
    mdl.eval()
    em, pm, ca_list, results = 0, 0, [], []
    for ex in test_examples:
        gold = normalize(ex['answer'])
        pred = greedy_generate(mdl, ex['prompt'])
        e    = int(pred == gold)
        p    = int(bool(pred and gold and (pred in gold or gold in pred)))
        g_ch = set(gold) - {' '}
        p_ch = set(pred) - {' '}
        ca   = len(g_ch & p_ch) / len(g_ch | p_ch) if g_ch | p_ch else 0.0
        em  += e; pm += p; ca_list.append(ca)
        results.append({'prompt': ex['prompt'], 'gold': gold, 'pred': pred,
                        'em': e, 'pm': p, 'char_acc': ca,
                        'template_id': ex.get('template_id', ''),
                        'attribute':   ex.get('attribute', '')})
    n = len(results)
    metrics = {'exact_match':  round(em / n, 4),
               'prefix_match': round(pm / n, 4),
               'char_set_acc': round(sum(ca_list) / n, 4), 'n': n}
    print(f'[{label}]  EM={metrics["exact_match"]*100:.2f}%  '
          f'PM={metrics["prefix_match"]*100:.2f}%  '
          f'CharAcc={metrics["char_set_acc"]*100:.2f}%  (n={n})')
    return metrics, results


test_examples = [json.loads(l) for l in open(DATA_DIR / 'test.jsonl', encoding='utf-8')][:500]

# Pre-trained baseline
pre_model = load_checkpoint(CKPT_PRE, cfg, device, dropout=0.0)
print('Evaluating PRE-TRAINED model ...')
pre_metrics, pre_results = evaluate(pre_model, test_examples, 'Pre-trained')

# Fine-tuned model (already loaded)
print('\nEvaluating FINE-TUNED model ...')
ft_metrics, ft_results = evaluate(model, test_examples, 'Fine-tuned')

  Loaded best_checkpoint.pth  (27.09M params)
Evaluating PRE-TRAINED model ...


[Pre-trained]  EM=0.00%  PM=0.40%  CharAcc=13.00%  (n=500)

Evaluating FINE-TUNED model ...


[Fine-tuned]  EM=54.00%  PM=54.00%  CharAcc=90.40%  (n=500)


## 6. Results Summary

In [7]:
print(f"\n{'Metric':<25} {'Pretrained':>12} {'Finetuned':>12} {'Delta':>10}")
print('-' * 62)
for k in ['exact_match', 'prefix_match', 'char_set_acc']:
    pv, fv = pre_metrics[k], ft_metrics[k]
    print(f'  {k:<23} {pv*100:>11.2f}% {fv*100:>11.2f}% {(fv-pv)*100:>+9.2f}%')

print('\nPer-template Exact Match:')
tmpl_pre = defaultdict(lambda: {'em': 0, 'n': 0})
tmpl_ft  = defaultdict(lambda: {'em': 0, 'n': 0})
for r in pre_results:
    tmpl_pre[r['template_id']]['em'] += r['em']; tmpl_pre[r['template_id']]['n'] += 1
for r in ft_results:
    tmpl_ft[r['template_id']]['em']  += r['em']; tmpl_ft[r['template_id']]['n']  += 1

for t in sorted(tmpl_ft):
    n = tmpl_ft[t]['n']
    p_acc = tmpl_pre[t]['em'] / max(tmpl_pre[t]['n'], 1) * 100
    f_acc = tmpl_ft[t]['em']  / max(n, 1) * 100
    print(f'  {t:<25}  pre: {p_acc:5.1f}%  ft: {f_acc:5.1f}%  (n={n})')


Metric                      Pretrained    Finetuned      Delta
--------------------------------------------------------------
  exact_match                    0.00%       54.00%    +54.00%
  prefix_match                   0.40%       54.00%    +53.60%
  char_set_acc                  13.00%       90.40%    +77.40%

Per-template Exact Match:
  t_direct_less              pre:   0.0%  ft:  41.5%  (n=82)
  t_direct_more              pre:   0.0%  ft:  45.3%  (n=86)
  t_multihop_largest         pre:   0.0%  ft:  58.2%  (n=67)
  t_multihop_smallest        pre:   0.0%  ft:  55.7%  (n=61)
  t_rank_three               pre:   0.0%  ft:  41.2%  (n=34)
  t_transitive_largest       pre:   0.0%  ft:  62.7%  (n=51)
  t_transitive_relation      pre:   0.0%  ft:  72.4%  (n=58)
  t_transitive_smallest      pre:   0.0%  ft:  59.0%  (n=61)


## 7. Attention Analysis — Entropy and Distance Across Layers

In [8]:
def analyze_attentions(mdl, sample_prompts):
    layer_entropies = defaultdict(list)
    layer_distances = defaultdict(list)
    for p in sample_prompts:
        ids = tokenizer.encode(p)[:128]
        x = torch.tensor([ids], dtype=torch.long, device=device)
        _, _, attns = mdl(x)
        T = len(ids)
        dist = torch.arange(T, device=device).unsqueeze(0) - torch.arange(T, device=device).unsqueeze(1)
        dist = dist.abs().float()
        for l_idx, attn in enumerate(attns):
            p_safe = attn.clamp(min=1e-9)
            ent = -(attn * torch.log(p_safe)).sum(dim=-1).mean().item()
            d   = (attn[0] * dist).sum(dim=-1).mean().item()
            layer_entropies[l_idx].append(ent)
            layer_distances[l_idx].append(d)
    return (
        [float(np.mean(layer_entropies[l])) for l in range(cfg['n_layers'])],
        [float(np.mean(layer_distances[l])) for l in range(cfg['n_layers'])],
    )

sample_p = [ex['prompt'] for ex in test_examples[:50]]
pre_ent, pre_dst = analyze_attentions(pre_model, sample_p)
ft_ent,  ft_dst  = analyze_attentions(model,     sample_p)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
layers = list(range(1, cfg['n_layers'] + 1))
ax1.plot(layers, pre_ent, 'o--', color='gray', label='Pre-trained')
ax1.plot(layers, ft_ent,  's-',  color='blue', label='Fine-tuned')
ax1.set_xlabel('Layer'); ax1.set_ylabel('Entropy (nats)')
ax1.set_title('Attention Entropy across Layers'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(layers, pre_dst, 'o--', color='gray', label='Pre-trained')
ax2.plot(layers, ft_dst,  's-',  color='red',  label='Fine-tuned')
ax2.set_xlabel('Layer'); ax2.set_ylabel('Mean Distance (tokens)')
ax2.set_title('Attention Distance across Layers'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'attention_analysis.png', dpi=150)
plt.show()
print('Saved:', RESULTS_DIR / 'attention_analysis.png')

Saved: /Users/nikhilsai/Documents/Courses/LMA/SelfProject/individual-project-nikk8076/nepali/results/attention_analysis.png


/var/folders/_q/gn6yzv0d07l_f_cgh3l13n1h0000gn/T/ipykernel_82733/3448424509.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
